In [1]:
import nltk

from nltk.corpus import stopwords

from nltk.tokenize import word_tokenize

from nltk.stem import WordNetLemmatizer


In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')


punkt is a pre-trained tokenizer used by NLTK to split text into:

    Sentences (sent_tokenize)
    
    Words (word_tokenize)

It knows things like:

    Periods don’t always end sentences (Dr., Mr.)
    
    How abbreviations work
    
    Sentence boundaries in English (and some other languages)
    

In [2]:

docs = [
    "Machine learning is a branch of artificial intelligence.",

    "Deep learning is a branch of machine learning.",
    
    "Quantum computers use quantum bits and qubits."
]

In [3]:

stop_words = set(stopwords.words('english'))

lemmatizer = WordNetLemmatizer()


In [4]:

texts = []
for doc in docs:
    tokens = word_tokenize(doc.lower())
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t.isalpha() and t not in stop_words]
    texts.append(tokens)

print(texts)


[['machine', 'learning', 'branch', 'artificial', 'intelligence'], ['deep', 'learning', 'branch', 'machine', 'learning'], ['quantum', 'computer', 'use', 'quantum', 'bit', 'qubits']]


In [5]:

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

vectorizer = CountVectorizer()
X = vectorizer.fit_transform([" ".join(t) for t in texts])

X

<3x11 sparse matrix of type '<class 'numpy.int64'>'
	with 14 stored elements in Compressed Sparse Row format>

In [7]:

lda = LatentDirichletAllocation(n_components=2, random_state=42)
lda.fit(X)


LatentDirichletAllocation(n_components=2, random_state=42)

In [8]:

feature_names = vectorizer.get_feature_names_out()
feature_names

array(['artificial', 'bit', 'branch', 'computer', 'deep', 'intelligence',
       'learning', 'machine', 'quantum', 'qubits', 'use'], dtype=object)

In [9]:

topics = [
    [feature_names[i] for i in topic.argsort()[-5:]]
    for topic in lda.components_
]

print(topics)


[['intelligence', 'deep', 'branch', 'machine', 'learning'], ['bit', 'computer', 'qubits', 'use', 'quantum']]


### Metrics


C_v measures how semantically consistent the top words of a topic are.

In simple terms:  Do the words in a topic actually belong together in meaning?


In [ ]:
How C_v is calculated (conceptually)

C_v combines 4 ideas:

1 Segmentation
    Break topic words into pairs

2 Probability estimation
    Measure how often word pairs co-occur in a sliding window

3 Normalized PMI (NPMI)
    Measures strength of association between words

4 Aggregation (cosine similarity)
    Produces one coherence score per topic


### gensim


CoherenceModel is a Gensim utility used to evaluate topic models by measuring how semantically coherent the words in each topic are.

Why We Need It

    Topic modelling has no accuracy
    
    Topics must be judged by interpretability
    
    CoherenceModel approximates human judgment


In [10]:

from gensim.corpora.dictionary import Dictionary
from gensim.models import CoherenceModel

dictionary = Dictionary(texts)

coherence_model = CoherenceModel(
    topics=topics,
    texts=texts,
    dictionary=dictionary,
    coherence='c_v'
)

coherence_score = coherence_model.get_coherence()
print("C_v Coherence:", coherence_score)


C_v Coherence: 0.8840479001119264


| C_v Score   | Interpretation                          |
| ----------- | --------------------------------------- |
| < 0.30      | Poor / noisy topics ❌                   |
| 0.30 – 0.50 | Acceptable                              |
| 0.50 – 0.65 | Good 👍                                 |
| 0.65 – 0.75 | Very good                               |
| **> 0.75**  | **Excellent / highly interpretable 🔥** |
